In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
model_path = "/content/drive/MyDrive/Colab Notebooks/model"

In [ ]:
!pip install shap lime captum -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 9.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 29.2 MB/s eta 0:00:00


In [ ]:
import json
import torch

# Check both config files
with open(f"{model_path}/config.json") as f:
    print("=== config.json ===")
    print(json.dumps(json.load(f), indent=2))

with open(f"{model_path}/model_config.json") as f:
    print("=== model_config.json ===")
    print(json.dumps(json.load(f), indent=2))

# Inspect actual keys/shapes in the checkpoint
state_dict = torch.load(f"{model_path}/pytorch_model.bin", map_location="cpu")
print("\n=== First 20 keys in pytorch_model.bin ===")
for k in list(state_dict.keys())[:20]:
    print(k, "->", state_dict[k].shape)

print(f"\nTotal keys: {len(state_dict.keys())}")

=== config.json ===
{
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 4096,
  "layer_norm_eps": 1e-07,
  "legacy": true,
  "max_position_embeddings": 512,
  "max_relative_positions": -1,
  "model_type": "deberta-v2",
  "norm_rel_ebd": "layer_norm",
  "num_attention_heads": 16,
  "num_hidden_layers": 24,
  "pad_token_id": 0,
  "pooler_dropout": 0.0,
  "pooler_hidden_act": "gelu",
  "pooler_hidden_size": 1024,
  "pos_att_type": [
    "p2c",
    "c2p"
  ],
  "position_biased_input": false,
  "position_buckets": 256,
  "relative_attention": true,
  "share_att_key": true,
  "tie_word_embeddings": true,
  "transformers_version": "5.16.1",
  "type_vocab_size": 0,
  "use_cache": false,
  "vocab_size": 128100
}
=== model_config.json ===
{
  "model_name": "microsoft/deberta-v3-large",
  "num_features": 12,
  

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
import json
import numpy as np

model_path = "/content/drive/MyDrive/Colab Notebooks/model"

# 1. Define the exact same class used during training
class DebertaGatedHybrid(nn.Module):
    def __init__(self, model_name, num_features=12, feature_dim=256):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(model_name, torch_dtype=torch.float32)
        hidden_size = self.transformer.config.hidden_size

        self.context_projection = nn.Sequential(
            nn.Linear(hidden_size, feature_dim), nn.ReLU(), nn.Dropout(0.2)
        )
        self.feature_projection = nn.Sequential(
            nn.Linear(num_features, feature_dim), nn.ReLU(), nn.Dropout(0.2)
        )
        self.gate = nn.Sequential(
            nn.Linear(feature_dim * 2, feature_dim), nn.Sigmoid()
        )
        self.fusion = nn.Sequential(
            nn.Linear(feature_dim, feature_dim), nn.ReLU(), nn.Dropout(0.3)
        )
        self.classifier = nn.Linear(feature_dim, 2)

    def forward(self, input_ids, attention_mask, nlp_features, labels=None):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        contextual_embedding = outputs.last_hidden_state[:, 0, :]
        context = self.context_projection(contextual_embedding)
        features = self.feature_projection(nlp_features.float())
        combined = torch.cat([context, features], dim=1)
        gate = self.gate(combined)
        fused = gate * context + (1 - gate) * features
        fused = self.fusion(fused)
        logits = self.classifier(fused)
        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels.long())
        return {"loss": loss, "logits": logits}

# 2. Read model_config.json to get exact params
with open(f"{model_path}/model_config.json") as f:
    mcfg = json.load(f)

print(mcfg)

# 3. Instantiate with the correct backbone name
model = DebertaGatedHybrid(
    model_name=mcfg["model_name"],       # "microsoft/deberta-v3-large"
    num_features=mcfg["num_features"],   # 12
    feature_dim=mcfg["context_projection_dimension"]  # 256
)

# 4. Load the saved weights
state_dict = torch.load(f"{model_path}/pytorch_model.bin", map_location="cpu")
model.load_state_dict(state_dict)
model.eval()
model.to("cuda" if torch.cuda.is_available() else "cpu")

print("✓ Model loaded successfully")

# 5. Load tokenizer + feature normalization stats
tokenizer = AutoTokenizer.from_pretrained(model_path)
feature_mean = np.load(f"{model_path}/feature_mean.npy")
feature_std = np.load(f"{model_path}/feature_std.npy")
with open(f"{model_path}/feature_names.json") as f:
    feature_names = json.load(f)

print("Feature names:", feature_names)

{'model_name': 'microsoft/deberta-v3-large', 'num_features': 12, 'num_labels': 2, 'max_length': 512, 'context_dimension': 1024, 'context_projection_dimension': 256, 'feature_projection_dimension': 256, 'fusion_dimension': 256, 'architecture': 'DeBERTa-v3-large + 12 NLP features + gated fusion', 'random_seed': 42}


config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  874MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  874MB            

model.safetensors: downloading bytes:           |  0.00B            

✓ Model loaded successfully
Feature names: ['urgency_term_count', 'credential_term_count', 'threat_term_count', 'financial_term_count', 'cta_term_count', 'url_count', 'email_count', 'phone_count', 'exclamation_count', 'question_count', 'uppercase_word_ratio', 'log_text_length']


In [ ]:
with open(f"{model_path}/test_results.json") as f:
    test_results = json.load(f)

print(test_results)

{'test_results': {'test_loss': 0.03715737536549568, 'test_accuracy': 0.9948630136986302, 'test_precision': 0.9969183359013868, 'test_recall': 0.9892966360856269, 'test_f1': 0.9930928626247122}, 'confusion_matrix': [[1096, 2], [7, 647]]}


In [ ]:
import pandas as pd

base_path = "/content/drive/MyDrive/Colab Notebooks"

train_df = pd.read_csv(f"{base_path}/phishing_train.csv")
val_df = pd.read_csv(f"{base_path}/phishing_validation.csv")
test_df = pd.read_csv(f"{base_path}/phishing_test.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)
print("\nColumns:", test_df.columns.tolist())

print("\n--- Sample rows ---")
print(test_df[["Email Text", "processed_text", "Email Type", "label"]].head(3))

Train: (14015, 4)
Validation: (1752, 4)
Test: (1752, 4)

Columns: ['Email Text', 'processed_text', 'Email Type', 'label']

--- Sample rows ---
                                          Email Text  \
0  draft doorstep budget attached is a draft door...   
1  On Mon, 05 Aug 2002 21:39:59 -0400 \nTom Reing...   
2  >>>>> "N" == Ned Jackson Lovely  writes:    N>...   

                                      processed_text  Email Type  label  
0  draft doorstep budget attached is a draft door...  Safe Email      0  
1  On Mon, 05 Aug 2002 21:39:59 -0400 \nTom Reing...  Safe Email      0  
2  >>>>> "N" == Ned Jackson Lovely writes: N> On ...  Safe Email      0  


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
import json

model_path = "/content/drive/MyDrive/Colab Notebooks/model"

class DebertaGatedHybrid(nn.Module):
    def __init__(self, model_name, num_features=12, feature_dim=256):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(model_name, torch_dtype=torch.float32)
        hidden_size = self.transformer.config.hidden_size
        self.context_projection = nn.Sequential(
            nn.Linear(hidden_size, feature_dim), nn.ReLU(), nn.Dropout(0.2)
        )
        self.feature_projection = nn.Sequential(
            nn.Linear(num_features, feature_dim), nn.ReLU(), nn.Dropout(0.2)
        )
        self.gate = nn.Sequential(
            nn.Linear(feature_dim * 2, feature_dim), nn.Sigmoid()
        )
        self.fusion = nn.Sequential(
            nn.Linear(feature_dim, feature_dim), nn.ReLU(), nn.Dropout(0.3)
        )
        self.classifier = nn.Linear(feature_dim, 2)

    def forward(self, input_ids, attention_mask, nlp_features, labels=None):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        contextual_embedding = outputs.last_hidden_state[:, 0, :]
        context = self.context_projection(contextual_embedding)
        features = self.feature_projection(nlp_features.float())
        combined = torch.cat([context, features], dim=1)
        gate = self.gate(combined)
        fused = gate * context + (1 - gate) * features
        fused = self.fusion(fused)
        logits = self.classifier(fused)
        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels.long())
        return {"loss": loss, "logits": logits}

with open(f"{model_path}/model_config.json") as f:
    mcfg = json.load(f)

model = DebertaGatedHybrid(
    model_name=mcfg["model_name"],
    num_features=mcfg["num_features"],
    feature_dim=mcfg["context_projection_dimension"]
)

state_dict = torch.load(f"{model_path}/pytorch_model.bin", map_location="cpu")
model.load_state_dict(state_dict)
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

tokenizer = AutoTokenizer.from_pretrained(model_path)
feature_mean = np.load(f"{model_path}/feature_mean.npy")
feature_std = np.load(f"{model_path}/feature_std.npy")

print("✓ Model, tokenizer, and feature stats loaded")

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Model, tokenizer, and feature stats loaded


In [ ]:
import re
import numpy as np

URGENCY_TERMS = ["urgent", "urgently", "immediately", "immediate", "now", "today",
                 "asap", "quickly", "deadline", "expire", "expired", "final", "action"]
CREDENTIAL_TERMS = ["password", "passwd", "username", "login", "credential", "credentials",
                     "verify", "verification", "authenticate", "authentication", "account"]
THREAT_TERMS = ["suspend", "suspended", "terminate", "terminated", "blocked", "block",
                "close", "closed", "penalty", "fraud", "unauthorized", "warning", "security"]
FINANCIAL_TERMS = ["payment", "pay", "invoice", "money", "bank", "transfer", "transaction",
                    "refund", "credit", "debit", "fee", "account", "billing"]
CTA_TERMS = ["click", "clicking", "visit", "open", "download", "confirm", "verify",
             "submit", "update", "activate", "login"]

def extract_nlp_features(text):
    text = str(text)
    words = re.findall(r"\b\w+\b", text.lower())

    urgency_count = sum(w in URGENCY_TERMS for w in words)
    credential_count = sum(w in CREDENTIAL_TERMS for w in words)
    threat_count = sum(w in THREAT_TERMS for w in words)
    financial_count = sum(w in FINANCIAL_TERMS for w in words)
    cta_count = sum(w in CTA_TERMS for w in words)
    url_count = len(re.findall(r"https?://\S+|www\.\S+", text, flags=re.IGNORECASE))
    email_count = len(re.findall(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b", text))
    phone_count = len(re.findall(r"\b(?:\+?\d[\d\s().-]{7,}\d)\b", text))
    exclamation_count = text.count("!")
    question_count = text.count("?")

    all_words = re.findall(r"\b[A-Za-z]+\b", text)
    uppercase_ratio = (sum(w.isupper() and len(w) > 1 for w in all_words) / len(all_words)
                        if all_words else 0.0)
    log_text_length = np.log1p(len(text))

    return [urgency_count, credential_count, threat_count, financial_count, cta_count,
            url_count, email_count, phone_count, exclamation_count, question_count,
            uppercase_ratio, log_text_length]

def get_normalized_features(text):
    raw = np.array(extract_nlp_features(text), dtype=np.float32)
    return (raw - feature_mean) / feature_std

In [ ]:
sample_text = test_df["processed_text"].iloc[0]
print(sample_text[:200])
print(get_normalized_features(sample_text))

draft doorstep budget attached is a draft doorstep budget for your review . i will be here next week to update it for your comments . regards shona
[-0.02778951 -0.03190457 -0.05564856 -0.02752902  0.01161071 -0.03020098
 -0.3075235  -0.01986818 -0.02249081 -0.03631876 -0.27510083 -1.5772076 ]


In [ ]:
from torch.utils.data import Dataset, DataLoader

class PhishingHybridDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=512):
        self.texts = df["processed_text"].tolist()
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        encoding = self.tokenizer(
            text, truncation=True, max_length=self.max_length,
            padding="max_length", return_tensors="pt"
        )
        nlp_feat = get_normalized_features(text)
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "nlp_features": torch.tensor(nlp_feat, dtype=torch.float32),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
            "idx": idx
        }

test_dataset = PhishingHybridDataset(test_df, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

all_preds = []
all_labels = []
all_probs = []
all_indices = []

model.eval()
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        nlp_features = batch["nlp_features"].to(device)
        labels = batch["labels"]

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, nlp_features=nlp_features)
        logits = outputs["logits"]
        probs = torch.softmax(logits, dim=1)[:, 1]  # phishing probability
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())
        all_indices.extend(batch["idx"].numpy())

import pandas as pd
results_df = pd.DataFrame({
    "test_idx": all_indices,
    "true_label": all_labels,
    "predicted_label": all_preds,
    "phishing_probability": all_probs
})
results_df["correct"] = results_df["true_label"] == results_df["predicted_label"]

print("Accuracy:", results_df["correct"].mean())
print("\nConfusion matrix check:")
from sklearn.metrics import confusion_matrix
print(confusion_matrix(all_labels, all_preds))

print("\nTotal errors:", (~results_df["correct"]).sum())

Accuracy: 0.9948630136986302

Confusion matrix check:
[[1096    2]
 [   7  647]]

Total errors: 9


In [ ]:
errors_df = results_df[~results_df["correct"]].copy()
errors_df["true_label_name"] = errors_df["true_label"].map({0: "Safe Email", 1: "Phishing Email"})
errors_df["predicted_label_name"] = errors_df["predicted_label"].map({0: "Safe Email", 1: "Phishing Email"})

# Attach the actual text
errors_df["text"] = errors_df["test_idx"].apply(lambda i: test_df["processed_text"].iloc[i])

# Sort false negatives (phishing missed) by confidence — most dangerous misses first
false_negatives = errors_df[
    (errors_df["true_label"] == 1) & (errors_df["predicted_label"] == 0)
].sort_values("phishing_probability")

false_positives = errors_df[
    (errors_df["true_label"] == 0) & (errors_df["predicted_label"] == 1)
].sort_values("phishing_probability", ascending=False)

print("=== FALSE NEGATIVES (phishing missed) ===")
print(false_negatives[["test_idx", "phishing_probability", "text"]].to_string())

print("\n=== FALSE POSITIVES (safe flagged as phishing) ===")
print(false_positives[["test_idx", "phishing_probability", "text"]].to_string())

=== FALSE NEGATIVES (phishing missed) ===
      test_idx  phishing_probability                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [ ]:
!pip install shap lime -q

import shap

def predict_proba_wrapper(texts):
    """SHAP-compatible prediction function returning class probabilities."""
    results = []
    for text in texts:
        encoding = tokenizer(text, truncation=True, max_length=512,
                              padding="max_length", return_tensors="pt").to(device)
        nlp_feat = torch.tensor(get_normalized_features(text), dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            outputs = model(input_ids=encoding["input_ids"],
                             attention_mask=encoding["attention_mask"],
                             nlp_features=nlp_feat)
            probs = torch.softmax(outputs["logits"], dim=1).cpu().numpy()
        results.append(probs[0])
    return np.array(results)

case_indices = [1150, 99, 1470, 717]
cases = test_df.loc[case_indices, "processed_text"].tolist()

masker = shap.maskers.Text(tokenizer=r"\W+")
explainer = shap.Explainer(predict_proba_wrapper, masker)

shap_values = explainer(cases[:1])  # start with just ONE case to test speed first
shap.plots.text(shap_values[0, :, 1])  # class 1 = phishing

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 2it [00:46, 46.73s/it]               


In [ ]:
for idx, text in zip([99, 1470, 717], cases[1:]):
    print(f"\n{'='*80}\nCase test_idx={idx}\n{'='*80}")
    sv = explainer([text])
    shap.plots.text(sv[0, :, 1])


Case test_idx=99


  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 2it [00:42, 42.44s/it]               



Case test_idx=1470


  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 2it [00:39, 39.51s/it]               



Case test_idx=717


  0%|          | 0/306 [00:00<?, ?it/s]

PartitionExplainer explainer: 2it [00:18, 18.54s/it]               


In [ ]:
import matplotlib.pyplot as plt
shap.plots.text(shap_values[0, :, 1], display=False)
# or for force/bar plots:
plt.savefig('/content/shap_case_1150.png', dpi=300, bbox_inches='tight')

<Figure size 640x480 with 0 Axes>

## Integrated Gradients (Captum) — Cross-Validation of SHAP Findings

SHAP treats the model as a black box and perturbs words to estimate importance. **Integrated Gradients** instead uses the model's actual gradients — computing how much each input token's embedding contributed to the output by integrating gradients along a path from a neutral baseline (all-padding input) to the real input. Because it uses the model's internals directly, it is generally considered a more faithful attribution method, and serves as a strong cross-check against the SHAP results above.

The same four error cases (test indices 1150, 99, 1470, 717) are re-explained here so the two methods can be compared side-by-side.

In [ ]:
!pip install captum -q

import gc
import torch
from captum.attr import LayerIntegratedGradients
import numpy as np


**Step 1 — Wrapper function and LayerIntegratedGradients setup.**

The model takes both `input_ids` and `nlp_features` together, so Integrated Gradients is applied only to the text-embedding path (`model.transformer.embeddings.word_embeddings`); the NLP feature branch is held fixed as an `additional_forward_arg` and is not attributed here (see Limitations).

**Baseline construction note:** the baseline keeps `[CLS]`/`[SEP]` (and any other special tokens) fixed to their real values, replacing only the content tokens with the pad token. Perturbing the special tokens as well was tested first and produced a large convergence delta together with `[CLS]`/`[SEP]` dominating the attribution rankings — an artifact of toggling tokens the model always expects to see, not a genuine finding about the email content.

In [ ]:
# Forward pass that returns only the phishing-class logit, for Captum to differentiate through
def model_forward(input_ids, attention_mask, nlp_features):
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, nlp_features=nlp_features)
    return outputs["logits"]

# Attribute at the transformer's word-embedding layer
lig = LayerIntegratedGradients(model_forward, model.transformer.embeddings.word_embeddings)


In [ ]:
def explain_with_ig(text, target_class=1, n_steps=200, internal_batch_size=4):
    # Clear cache before each run to avoid leftover memory from previous calls
    torch.cuda.empty_cache()
    gc.collect()

    encoding = tokenizer(text, truncation=True, max_length=512,
                         padding="max_length", return_tensors="pt").to(device)
    input_ids = encoding["input_ids"]
    attention_mask = encoding["attention_mask"]
    nlp_feat = torch.tensor(get_normalized_features(text), dtype=torch.float32).unsqueeze(0).to(device)

    # Baseline: replace only content tokens with the pad token.
    # Keep [CLS] / [SEP] (and any other special tokens) fixed, since the model
    # always expects them - perturbing them adds noise and inflates convergence error.
    special_ids = set(tokenizer.all_special_ids)
    baseline_ids = input_ids.clone()
    content_mask = torch.tensor(
        [[tok_id not in special_ids for tok_id in seq] for seq in input_ids.tolist()],
        device=device
    )
    baseline_ids[content_mask] = tokenizer.pad_token_id

    with torch.no_grad():
        input_logit = model_forward(input_ids, attention_mask, nlp_feat)[0, target_class].item()
        baseline_logit = model_forward(baseline_ids, attention_mask, nlp_feat)[0, target_class].item()
    score_diff = input_logit - baseline_logit

    attributions, delta = lig.attribute(
        inputs=input_ids,
        baselines=baseline_ids,
        additional_forward_args=(attention_mask, nlp_feat),
        target=target_class,
        n_steps=n_steps,
        internal_batch_size=internal_batch_size,  # chunks the n_steps interpolations to avoid OOM
        return_convergence_delta=True
    )

    # Sum attributions across the embedding dimension -> one score per token
    token_scores = attributions.sum(dim=-1).squeeze(0)
    raw_token_scores = token_scores.detach().cpu().numpy().copy()
    token_scores = token_scores / torch.norm(token_scores)  # normalize for readability

    tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze(0))

    # Free the attribution tensor explicitly
    del attributions
    torch.cuda.empty_cache()

    meta = {
        "delta": delta.item(),
        "score_diff": score_diff,
        "relative_delta": abs(delta.item()) / (abs(score_diff) + 1e-8)
    }
    return list(zip(tokens, token_scores.detach().cpu().numpy())), meta


# Punctuation / whitespace-only subword pieces to exclude from the *readable* ranking.
# These carry little independent interpretable meaning even when their raw score is large.
PUNCT_ONLY = {",", ".", "-", "'", '"', ":", ";", "!", "?", "(", ")", "▁", "▁,", "▁.", "▁-"}

def print_ig_result(tokens_scores, meta, top_k=25, hide_special=True, hide_punct=True):
    print(f"Convergence delta: {meta['delta']:.4f}   "
          f"Output score diff (input - baseline): {meta['score_diff']:.4f}   "
          f"Relative delta: {meta['relative_delta']:.2%}")
    print("(Relative delta is the more meaningful check for DeBERTa-style relative-attention "
          "models - see the markdown note above on why the raw delta does not shrink to ~0.)\n")

    special_display = {"[PAD]", "<pad>"}
    if hide_special:
        special_display |= {"[CLS]", "[SEP]", "<s>", "</s>"}
    filtered = [(t, s) for t, s in tokens_scores if t not in special_display]
    if hide_punct:
        filtered = [(t, s) for t, s in filtered if t.strip("▁") not in PUNCT_ONLY and t not in PUNCT_ONLY]

    sorted_by_abs = sorted(filtered, key=lambda x: abs(x[1]), reverse=True)[:top_k]
    print(f"{'Token':<20} {'Score':>10}  Direction")
    print("-" * 45)
    for tok, score in sorted_by_abs:
        direction = "-> phishing" if score > 0 else "-> safe"
        print(f"{tok:<20} {score:>10.4f}  {direction}")


**Step 2 — Run on Case 1 (test index 1150) first**, to confirm convergence and timing before scaling up to all four cases. `convergence_delta` is Captum's built-in sanity check and should be small (close to 0); if it is large, increase `n_steps` (already raised to 100 above).

In [ ]:
case_text_1150 = test_df.loc[1150, "processed_text"]
result_1150, meta_1150 = explain_with_ig(case_text_1150)
print_ig_result(result_1150, meta_1150)


**Step 3 — Run on the remaining three cases (99, 1470, 717)** for the full side-by-side set.

In [ ]:
ig_results = {1150: (result_1150, meta_1150)}

for idx in [99, 1470, 717]:
    text = test_df.loc[idx, "processed_text"]
    print(f"\n{'='*80}\nCase test_idx={idx}\n{'='*80}")
    result, meta = explain_with_ig(text)
    print_ig_result(result, meta)
    ig_results[idx] = (result, meta)


**Step 4 — SHAP vs. Integrated Gradients agreement check.**

For each case, this pulls the top tokens each method flagged as most influential and shows them side-by-side. Strong overlap between the two independent methods meaningfully strengthens confidence in the explanation; disagreement is also a useful finding worth discussing in the report.

In [ ]:
def top_ig_tokens(idx, k=10, hide_punct=True):
    tokens_scores, _ = ig_results[idx]
    special_display = {"[PAD]", "<pad>", "[CLS]", "[SEP]", "<s>", "</s>"}
    filtered = [(t, s) for t, s in tokens_scores if t not in special_display]
    if hide_punct:
        filtered = [(t, s) for t, s in filtered if t.strip("▁") not in PUNCT_ONLY and t not in PUNCT_ONLY]
    ranked = sorted(filtered, key=lambda x: abs(x[1]), reverse=True)[:k]
    return [t for t, s in ranked]

for idx in [1150, 99, 1470, 717]:
    print(f"\nCase {idx} — top Integrated Gradients tokens:")
    print(top_ig_tokens(idx))
    print("Compare this list against the red/blue SHAP tokens highlighted for the same case above.")


### Limitations of this Integrated Gradients pass

- Attribution is computed only over the text/embedding branch (`word_embeddings`); it does not directly attribute importance to the 12 handcrafted NLP features fed through the separate feature branch. As with the SHAP analysis, a companion feature-level check (e.g. inspecting the raw feature values and the learned gate weight for each case) would be needed to fully attribute the fused decision.
- Results depend on the choice of baseline (here, an all-padding input) and `n_steps`; both were kept consistent across the four cases for comparability.
- As with SHAP, token-level attribution reflects correlational importance to the model's output, not a claim about the true causal reasoning process.
- **Convergence delta remained non-trivial even after fixing the baseline and raising `n_steps` to 200.** This is expected for DeBERTa-v3: its disentangled/relative-position attention introduces nonlinearity along the straight-line embedding path that standard Integrated Gradients assumes, so the approximation does not converge as tightly as it would for a model with only absolute position embeddings (e.g. DistilBERT). Rather than treating this as invalidating the results, the *relative* delta (convergence delta as a fraction of the actual output score difference between input and baseline) is reported alongside the raw delta, and is the more appropriate way to judge attribution quality for this architecture. This should be reported as a documented methodological caveat rather than silently omitted.